In [1]:
!pip install mlflow boto3 awscli 

In [2]:
import setup_up
import os 
print(os.getenv("AWS_DEFAULT_REGION"))
print(bool(os.getenv("AWS_ACCESS_KEY_ID")))
print(bool(os.getenv("AWS_SECRET_ACCESS_KEY")))

eu-north-1
True
True


In [3]:
!aws sts get-caller-identity > /dev/null 2>&1

In [4]:
import mlflow
import setup_up
SERVER_URL = os.environ["SERVER_URL"]

mlflow.set_tracking_uri(SERVER_URL)

In [5]:
mlflow.set_experiment('Exp 4 - Handling imbalanced data')

/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/03/14 23:53:21 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - Handling imbalanced data' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/grayfog/YTSentiment/notebooks/ec2-13-51-197-16.eu-north-1.compute.amazonaws.com/273407204227159456', creation_time=1773528801841, experiment_id='273407204227159456', last_update_time=1773528801841, lifecycle_stage='active', name='Exp 4 - Handling imbalanced data', tags={}>

In [6]:
!pip install imblearn

In [7]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os


In [8]:
df = pd.read_csv('../data/read_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 3)

In [9]:
def run_imbalanced_experiment(imbalanced_method):
    ngram_range = (1,3)
    max_features = 1000

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, 
                                                        random_state=42, stratify=df['category'])

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    if imbalanced_method == 'class weights':
        class_weight = 'Balanced'
    else:
        class_weight = None

    if imbalanced_method == 'oversampling':
        smote = SMOTE(random_state=42)
        X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)
    elif imbalanced_method == 'adasyn':
        adasyn = ADASYN(random_state=42)
        X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)
    elif imbalanced_method == 'undersampling':
        rus = RandomUnderSampler(random_state=42)
        X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)
    elif imbalanced_method == 'smote_enn':
        smote_enn = SMOTEENN(random_state=42)
        X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    with mlflow.start_run() as run:
        mlflow.set_tag('mlflow.runName', f'Imbalanced_{imbalanced_method}_RandomForest_TFIDF_Trigrams')
        mlflow.set_tag('experiment_type','imbalanced_handling')
        mlflow.set_tag('model_type', 'RandomForestClassifier')

        mlflow.set_tag('description', f'RandomForest with TF-IDF Trigrams, imbalanced handling method = {imbalanced_method}')
        
        mlflow.log_param('vectorize_type','TF-IDF')
        mlflow.log_param('ngram_range', ngram_range)
        mlflow.log_param('imbalanced_method', imbalanced_method)

        n_estimators = 200
        max_depth = 15
        
        mlflow.log_param('n_estimators', n_estimators)
        mlflow.log_param('max_depth', max_depth)
        mlflow.log_param('imbalanced_method', imbalanced_method)
        
        # initialize and train the model 
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)
        model.fit(X_train_vec, y_train)

        y_pred = model.predict(X_test_vec)
    
        # log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
    
        # log classification report 
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics,dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}', value)

        conf_matrix = confusion_matrix(y_test,y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
        plt.xlabel = 'Predicted'
        plt.ylable = 'Actual'
        plt.title = f'Confusion Matrix:TD-IDF_max_features=1000, imbalance_method={imbalanced_method}'
        conf_matrix_filename = f'confusion_matrix_{imbalanced_method}.png'
        plt.savefig(conf_matrix_filename)
        mlflow.log_artifact(conf_matrix_filename)
        plt.close()

        mlflow.sklearn.log_model(model, name=f'random_forest_model_tfidf_trigrams_imbalanced_{imbalanced_method}')

imbalanced_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']

for method in imbalanced_methods:
    run_imbalanced_experiment(method)

/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/pyth